In [ ]:
"""
실시간 심박수 + 호흡수 동시 측정
  - 심박수 : open-rppg (rPPG 기반)
  - 호흡수 : EVM Motion 모드 (얼굴 미세 움직임 기반)

설치:
    pip install open-rppg
    pip install opencv-python numpy scipy
"""

import cv2
import numpy as np
import time
import argparse
from collections import deque
from scipy.signal import butter, lfilter
import rppg


# ══════════════════════════════════════════════════════
#  EVM 호흡수 파이프라인
# ══════════════════════════════════════════════════════

def build_laplacian_pyramid(frame, levels=4):
    # 1. 가우시안 피라미드 먼저 만들기 (흐리게 + 축소)
    g_pyr = [frame]               # Level 0 = 원본
    for _ in range(levels):
        frame = cv2.pyrDown(frame) # 흐리게 만들고 절반 크기로 축소
        g_pyr.append(frame)

    # 2. 라플라시안 = 원본 - 흐린버전 (엣지/움직임만 남음)
    l_pyr = []
    for i in range(levels, 0, -1):
        expanded = cv2.pyrUp(g_pyr[i])         # 작은 것을 다시 확대
        laplacian = g_pyr[i-1] - expanded      # 빼면 움직임 성분만 남음
        l_pyr.append(laplacian)

    return l_pyr  # pyr[0] = 가장 섬세한 움직임 성분


def butter_bandpass(low, high, fps, order=1):
    nyq = fps / 2.0
    lo = max(0.001, min(low / nyq, 0.999))
    hi = max(0.001, min(high / nyq, 0.999))
    if lo >= hi:
        hi = min(lo + 0.05, 0.999)
    b, a = butter(order, [lo, hi], btype='band')
    return b, a


def temporal_filter(buffer, b, a):
    """버퍼(deque) 전체에 버터워스 필터 적용 후 마지막 값 반환"""
    arr = np.stack(list(buffer), axis=0).astype(np.float32)  # (N, H, W, C)
    filtered = np.zeros_like(arr)
    for c in range(arr.shape[3]):
        for h in range(arr.shape[1]):
            filtered[:, h, :, c] = lfilter(b, a, arr[:, h, :, c], axis=0)
    return filtered[-1]


def compute_bpm(signal_buffer, fps, freq_low, freq_high):
    """
    1D 시계열 신호 → FFT → 주파수 피크 → BPM 반환
    반환: (bpm, confidence 0~1)
    """
    sig = np.array(signal_buffer)
    if len(sig) < 16:
        return 0.0, 0.0

    sig = sig - np.mean(sig)
    sig *= np.hanning(len(sig))

    fft_vals = np.abs(np.fft.rfft(sig, n=len(sig) * 4))
    freqs    = np.fft.rfftfreq(len(sig) * 4, d=1.0 / fps)

    mask = (freqs >= freq_low) & (freqs <= freq_high)
    if not np.any(mask):
        return 0.0, 0.0

    band_vals  = fft_vals[mask]
    band_freqs = freqs[mask]
    peak_idx   = np.argmax(band_vals)
    peak_freq  = band_freqs[peak_idx]

    confidence = float(band_vals[peak_idx]) / (np.sum(fft_vals) + 1e-9)
    confidence = min(confidence * 10.0, 1.0)

    return peak_freq * 60.0, confidence


class RespiratoryEVM:
    """
    EVM Motion 모드 기반 실시간 호흡수 측정기

    Parameters
    ----------
    fps         : 카메라 FPS
    freq_low    : 호흡 주파수 하한 (Hz), 기본 0.15 = 9 bpm
    freq_high   : 호흡 주파수 상한 (Hz), 기본 0.60 = 36 bpm
    levels      : 라플라시안 피라미드 레벨
    buf_size    : EVM 프레임 버퍼 크기
    sig_size    : FFT용 신호 버퍼 크기
    fixed_size  : ROI 리사이즈 고정 크기 (shape 통일용)
    """

    FREQ_LOW  = 0.15   # Hz (9 bpm)
    FREQ_HIGH = 0.60   # Hz (36 bpm)

    def __init__(self, fps=30.0, levels=4, buf_size=256,
                 sig_size=512, fixed_size=(64, 64)):
        self.fps        = fps
        self.levels     = levels
        self.fixed_size = fixed_size

        self.frame_buf = deque(maxlen=buf_size)
        self.sig_buf   = deque(maxlen=sig_size)

        self.b, self.a = butter_bandpass(
            self.FREQ_LOW, self.FREQ_HIGH, fps
        )

        # 결과
        self.rr_bpm  = 0.0
        self.rr_conf = 0.0

    def push(self, roi_bgr):
        """
        BGR ROI 프레임을 받아 신호 추출.
        충분히 쌓이면 self.rr_bpm / self.rr_conf 갱신.
        """
        # 고정 크기 리사이즈 (shape 불일치 방지)
        roi = cv2.resize(roi_bgr, self.fixed_size,
                         interpolation=cv2.INTER_LINEAR)
        roi_f = roi.astype(np.float32) / 255.0

        # 라플라시안 피라미드 첫 번째 레벨 (엣지 성분)
        pyr    = build_laplacian_pyramid(roi_f, self.levels)
        target = pyr[0]

        self.frame_buf.append(target)

        min_frames = self.frame_buf.maxlen // 2
        if len(self.frame_buf) < min_frames:
            return

        # 템포럴 필터링 → 증폭 → 신호 추출
        filtered = temporal_filter(self.frame_buf, self.b, self.a)
        self.sig_buf.append(float(np.mean(np.abs(filtered))))

    def update_bpm(self):
        """sig_buf가 충분히 차면 BPM 재계산"""
        if len(self.sig_buf) < 32:
            return
        self.rr_bpm, self.rr_conf = compute_bpm(
            self.sig_buf, self.fps, self.FREQ_LOW, self.FREQ_HIGH
        )
        # 생리적 범위 클리핑
        if self.rr_bpm < 6 or self.rr_bpm > 40:
            self.rr_conf *= 0.3   # 범위 벗어나면 신뢰도 낮춤

    def ready(self):
        return len(self.frame_buf) >= self.frame_buf.maxlen // 2


# ══════════════════════════════════════════════════════
#  HUD 그리기 유틸
# ══════════════════════════════════════════════════════

def draw_hud(frame, hr_bpm, hr_conf, rr_bpm, rr_conf,
             fps, hr_ready, rr_ready):
    h, w = frame.shape[:2]

    # 반투명 배경 패널
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (310, 185), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    # 색상: 신뢰도 낮으면 회색, 높으면 색상
    hr_color = (80,  220, 80)  if (hr_conf > 0.3 and hr_ready) else (130, 130, 130)
    rr_color = (80,  180, 255) if (rr_conf > 0.3 and rr_ready) else (130, 130, 130)

    # ── 심박수 ────────────────────────────────────────
    cv2.putText(frame, "HEART RATE  (rPPG)",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.52, hr_color, 1)
    hr_txt = f"{hr_bpm:.0f} BPM" if hr_ready else "warming up..."
    cv2.putText(frame, hr_txt,
                (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 1.3, hr_color, 2)
    # 신뢰도 바
    bar_w = int(min(hr_conf, 1.0) * 90)
    cv2.rectangle(frame, (10, 72), (100, 80), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, 72), (10 + bar_w, 80), hr_color, -1)

    # ── 호흡수 ────────────────────────────────────────
    cv2.putText(frame, "RESP RATE   (EVM)",
                (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.52, rr_color, 1)
    rr_txt = f"{rr_bpm:.0f} BPM" if rr_ready else "warming up..."
    cv2.putText(frame, rr_txt,
                (10, 145), cv2.FONT_HERSHEY_SIMPLEX, 1.3, rr_color, 2)
    # 신뢰도 바
    bar_w = int(min(rr_conf, 1.0) * 90)
    cv2.rectangle(frame, (10, 155), (100, 163), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, 155), (10 + bar_w, 163), rr_color, -1)

    # ── FPS ───────────────────────────────────────────
    cv2.putText(frame, f"FPS: {fps:.0f}",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (160, 160, 160), 1)

    # ── 정상 범위 안내 ────────────────────────────────
    cv2.putText(frame, "HR:60-100  RR:12-20",
                (w - 185, h - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.42, (120, 120, 120), 1)

    return frame


# ══════════════════════════════════════════════════════
#  메인 루프
# ══════════════════════════════════════════════════════

def run(camera_id=0):
    """
    open-rppg 심박수 + EVM 호흡수 동시 측정 메인 루프

    종료: Q 또는 ESC
    """

    # ── open-rppg 모델 초기화 ─────────────────────────
    print("[INFO] open-rppg 모델 로딩 중...")
    model = rppg.Model('ME-flow.rlap')
    print("[INFO] 모델 로딩 완료")

    # ── FPS 측정용 ────────────────────────────────────
    fps_timer   = time.time()
    frame_cnt   = 0
    measured_fps = 30.0

    # ── 호흡수 EVM ────────────────────────────────────
    rr_evm = RespiratoryEVM(fps=measured_fps)

    # ── BPM 갱신 타이머 ───────────────────────────────
    last_bpm_time = time.time()
    BPM_INTERVAL  = 2.0   # 초

    # ── 결과 변수 ─────────────────────────────────────
    hr_bpm, hr_conf = 0.0, 0.0
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    last_face = None

    print("=" * 50)
    print("  실시간 심박수 + 호흡수 측정 시작")
    print("  Q / ESC : 종료")
    print("=" * 50)

    # ── open-rppg 비디오 캡처 컨텍스트 진입 ─────────
    with model.video_capture(camera_id):
        for frame_rgb, box in model.preview:
            # open-rppg는 RGB 반환 → BGR로 변환
            frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            # ── FPS 측정 ──────────────────────────────
            frame_cnt += 1
            elapsed = time.time() - fps_timer
            if elapsed >= 2.0:
                measured_fps = frame_cnt / elapsed
                frame_cnt    = 0
                fps_timer    = time.time()

            # ── 얼굴 감지 (EVM ROI용) ─────────────────
            gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(
                gray, scaleFactor=1.1,
                minNeighbors=5, minSize=(80, 80)
            )
            if len(faces) > 0:
                last_face = max(faces, key=lambda f: f[2] * f[3])

            # ── open-rppg ROI 박스 그리기 ─────────────
            if box is not None:
                y1, y2 = box[0]
                x1, x2 = box[1]
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (80, 220, 80), 2)

            # ── EVM ROI 추출 + 호흡 신호 push ─────────
            if last_face is not None:
                fx, fy, fw, fh = last_face
                # 이마~코 영역
                ry1 = fy + int(fh * 0.05)
                ry2 = fy + int(fh * 0.65)
                rx1 = fx + int(fw * 0.15)
                rx2 = fx + int(fw * 0.85)

                roi = frame[ry1:ry2, rx1:rx2]
                if roi.size > 0:
                    rr_evm.push(roi)

                # EVM ROI 박스 표시 (노란색)
                cv2.rectangle(frame, (rx1, ry1), (rx2, ry2),
                              (0, 220, 220), 1)

            # ── BPM 갱신 (2초마다) ────────────────────
            now = time.time()
            if now - last_bpm_time >= BPM_INTERVAL:

                # 심박수: open-rppg
                result = model.hr(start=-10)
                if result and result.get('hr'):
                    hr_bpm  = float(result['hr'])
                    hr_conf = 0.9   # open-rppg는 자체 신뢰도 미제공
                    # 범위 벗어나면 신뢰도 낮춤
                    if hr_bpm < 40 or hr_bpm > 200:
                        hr_conf = 0.1

                # 호흡수: EVM
                rr_evm.update_bpm()

                # 콘솔 출력
                hr_str = f"{hr_bpm:.1f} BPM" if hr_bpm > 0 else "측정 중..."
                rr_str = f"{rr_evm.rr_bpm:.1f} BPM" if rr_evm.rr_bpm > 0 else "측정 중..."
                print(f"[심박수] {hr_str:12s}  |  [호흡수] {rr_str}")

                last_bpm_time = now

            # ── HUD 오버레이 ──────────────────────────
            frame = draw_hud(
                frame,
                hr_bpm=hr_bpm,
                hr_conf=hr_conf,
                rr_bpm=rr_evm.rr_bpm,
                rr_conf=rr_evm.rr_conf,
                fps=measured_fps,
                hr_ready=(hr_bpm > 0),
                rr_ready=rr_evm.ready(),
            )

            cv2.imshow("Vital Monitor  (HR: rPPG | RR: EVM)", frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                print("[INFO] 종료")
                break

    cv2.destroyAllWindows()


# ══════════════════════════════════════════════════════
#  진입점
# ══════════════════════════════════════════════════════

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="심박수 + 호흡수 실시간 측정")
    parser.add_argument("--camera", type=int, default=0,
                        help="카메라 ID (기본: 0)")
    args, _ = parser.parse_known_args()
    run(camera_id=args.camera)


[INFO] open-rppg 모델 로딩 중...
[INFO] 모델 로딩 완료
  실시간 심박수 + 호흡수 측정 시작
  Q / ESC : 종료
[심박수] 측정 중...       |  [호흡수] 측정 중...
[심박수] 73.6 BPM      |  [호흡수] 측정 중...
[심박수] 74.0 BPM      |  [호흡수] 측정 중...
[심박수] 74.2 BPM      |  [호흡수] 측정 중...
[심박수] 74.5 BPM      |  [호흡수] 32.7 BPM
[심박수] 73.3 BPM      |  [호흡수] 22.5 BPM
[심박수] 72.0 BPM      |  [호흡수] 21.6 BPM
[심박수] 71.9 BPM      |  [호흡수] 17.6 BPM


Exception in thread Thread-17:
Traceback (most recent call last):
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 950, in _bootstrap_inner
    self.run()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 888, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    

[INFO] 종료
